# NGB v4 small 4×4 comparison

This notebook discovers the intersection of **completed matched seeds** across
SGD, AdamW, and Muon. It reports final and validation-selected metrics, paired
optimizer contrasts, late-horizon drift, corrected perplexity intervals, and
block-resolved WeightWatcher trajectories. Blocks are never statistical
replicates.


In [ ]:
CONFIG_PATH = "configs/v4_small_4x4.yaml"
SEEDS = ""
NGB_STORAGE_ROOT = "/tmp/rg-ngb"


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / "baseline" / "ngb"]
NGB_ROOT_DIR = next(
    (path for path in candidates if (path / "configs" / "v4_one_head.yaml").is_file()),
    None,
)
if NGB_ROOT_DIR is None:
    raise FileNotFoundError("Run from baseline/ngb or the repository root")
RUNTIME_SRC = NGB_ROOT_DIR.parent / "nanogpt_one_head" / "src"
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

import json
import math
import matplotlib.pyplot as plt
import numpy as np

from rg_nanogpt_one_head import (
    OPTIMIZER_COLORS,
    OPTIMIZER_LABELS,
    SUPPORTED_OPTIMIZERS,
    discover_matched_complete_seeds,
    final_test_summary,
    load_config,
    load_epoch_metrics,
    load_layer_metrics,
    load_metrics,
    load_spectral_summary,
    load_test_results,
    paired_test_differences,
    plot_epoch_metric,
    plot_layer_metric,
    plot_spectral_optimizer_summary,
    run_diagnostics_table,
    run_slug,
    run_status_table,
    summarize_run_diagnostics,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


In [ ]:
config_path = (NGB_ROOT_DIR / CONFIG_PATH).resolve()
cfg = load_config(config_path)
results_root = Path(NGB_STORAGE_ROOT) / "results" / run_slug(cfg)
plot_root = Path(NGB_STORAGE_ROOT) / "plots" / run_slug(cfg) / "comparison"
plot_root.mkdir(parents=True, exist_ok=True)
optimizers = tuple(SUPPORTED_OPTIMIZERS)
seeds = (
    tuple(int(value.strip()) for value in SEEDS.split(",") if value.strip())
    if SEEDS.strip()
    else discover_matched_complete_seeds(results_root, optimizers=optimizers)
)
if not seeds:
    raise RuntimeError(f"No complete matched optimizer seeds under {results_root}")
print("config:", config_path)
print("results:", results_root)
print("matched complete seeds:", seeds)
display(run_status_table(results_root, optimizers=optimizers, seeds=seeds))

config_rows = []
for optimizer in optimizers:
    config_rows.append({
        "optimizer": optimizer,
        "optimizer_label": OPTIMIZER_LABELS[optimizer],
        **cfg["optimizer_profiles"][optimizer],
    })
config_table = pd.DataFrame(config_rows)
display(pd.DataFrame([{"run_slug": run_slug(cfg), **cfg["model"], **cfg["training"]}]))
display(config_table)
config_table.to_csv(plot_root / "optimizer_configurations.csv", index=False)

metrics = load_metrics(results_root, optimizers=optimizers, seeds=seeds)
epoch_metrics = load_epoch_metrics(results_root, optimizers=optimizers, seeds=seeds)
layer_metrics = load_layer_metrics(results_root, optimizers=optimizers, seeds=seeds)
spectral_summary = load_spectral_summary(results_root, optimizers=optimizers, seeds=seeds)
test_results = load_test_results(results_root, optimizers=optimizers, seeds=seeds)
print("rows:", {"metrics": len(metrics), "epoch": len(epoch_metrics), "layers": len(layer_metrics)})


In [ ]:
diagnostics = run_diagnostics_table(metrics, test_results)
diagnostic_summary = summarize_run_diagnostics(diagnostics)
summary = final_test_summary(test_results)
paired = paired_test_differences(test_results, optimizers=optimizers)

for frame, name in (
    (diagnostics, "run_diagnostics.csv"),
    (diagnostic_summary, "run_diagnostics_summary_95ci.csv"),
    (summary, "final_and_validation_selected_95ci.csv"),
    (paired, "paired_optimizer_differences_95ci.csv"),
):
    frame.to_csv(plot_root / name, index=False)

display(diagnostics.sort_values(["optimizer", "seed"]))
display(diagnostic_summary.sort_values(["metric", "optimizer"]))
display(summary.sort_values(["checkpoint", "metric", "optimizer"]))
display(paired.sort_values(["checkpoint", "metric", "contrast"]))


In [ ]:
for metric in (
    "train_loss", "val_loss", "test_loss",
    "train_accuracy", "val_accuracy", "test_accuracy",
    "val_generalization_gap", "test_generalization_gap",
    "weight_norm", "update_to_weight_ratio",
):
    if metric not in epoch_metrics.columns:
        continue
    plot_epoch_metric(
        epoch_metrics,
        metric=metric,
        optimizers=optimizers,
        title=f"{run_slug(cfg)}: {metric} (95% Student-t CI)",
        output=plot_root / f"{metric}.png",
    )
    plt.show()

# Late-horizon zooms make drift after the first half epoch explicit.
late = epoch_metrics[epoch_metrics["nominal_epoch"].astype(float) >= 0.5]
for metric in ("val_loss", "test_loss", "val_accuracy", "test_accuracy"):
    plot_epoch_metric(
        late,
        metric=metric,
        optimizers=optimizers,
        title=f"{run_slug(cfg)}: {metric}, epoch >= 0.5",
        output=plot_root / f"zoom_epoch_0p5_{metric}.png",
    )
    plt.show()


In [ ]:
for metric in ("alpha_median", "ERG_gap_median", "num_traps_mean"):
    plot_spectral_optimizer_summary(
        spectral_summary,
        metric=metric,
        optimizers=optimizers,
        output=plot_root / f"spectral_{metric}.png",
    )
    if metric == "alpha_median":
        plt.axhline(2.0, color="black", linestyle="--", linewidth=1.0)
    if metric == "ERG_gap_median":
        plt.axhline(0.0, color="black", linestyle="--", linewidth=1.0)
    plt.show()

for optimizer in optimizers:
    for metric in ("alpha", "ERG_gap", "num_traps"):
        plot_layer_metric(
            layer_metrics,
            optimizer=optimizer,
            metric=metric,
            title=f"{run_slug(cfg)} {OPTIMIZER_LABELS[optimizer]}: block-resolved {metric}",
            output=plot_root / f"{optimizer}_block_resolved_{metric}.png",
        )
        plt.show()
